Generalised Profiling Tutorial 

Lotka-Volterra Predator-Prey System

Alexander Johnston
Queensland University of Technology
a44.johnston@qut.edu.au

In [ ]:
using Plots, DelimitedFiles, DifferentialEquations, SparseArrays, Interpolations, Random, Distributions, NLopt, Dierckx, LaTeXStrings, BSplineKit, Plots.PlotMeasures, LinearAlgebra
gr()

plot_font = "Computer Modern"
default(fontfamily=plot_font,linewidth=1,framestyle=:box,label=nothing,grid=true)

#Model parameters
alpha = 1;
beta = 1;

a = [alpha, beta];

#Initial conditions
x0 = 1;
y0 = 0.5;

#Model
a = [alpha, beta];

#Number of synthetic data points
N_data = 41;

#Ratio of number of points in the spline grid to the number of points in the synthetic data set 
grid_ratio = 10;

#Number of B-Splines
df = N_data - 1;

#Discretisation grid size for spline
N_t = grid_ratio*df;

tmax = 15
tt=LinRange(0,tmax,N_t);
t_start = 0;
t_end = 15;
tt = LinRange(t_start, t_end, N_t);

#Standard deviation of additive Gaussian noise used to generate the synthetic data
sigma=0.05; 



In [ ]:
#Create synthetic data for the Lotka-Volterra predator-prey model.
#Note that this code block allows the user to generate new data synthetic data sets. However, the calculations
#conducted in the remaining code blocks use the same synthetic data set used for the paper to show the generation of
#plots identical to those produced in the paper. These data are loaded in the next cell.

#Lotka-Volterra predator-prey equations
function Lotka_Volterra!(du,u,a,t)
alpha = a[1];
beta = a[2];
du[1]=alpha*u[1] - u[1]*u[2]; 
du[2]=beta*u[1]*u[2] - u[2];
end

function odesolver(t,ic,a)
tspan=(0.0,maximum(t))
prob=ODEProblem(Lotka_Volterra!,ic,tspan, a)
alg=Tsit5()
sol=solve(prob,alg,saveat=t);
return sol
end

ic=[x0, y0]
sol = odesolver(tt,ic,a)
    
x = sol[1,:];
y = sol[2,:];

t = [0]
for (i, t_i) in enumerate(tt)
    if i%grid_ratio == 0
        t = vcat(t, t_i)
    end
end
        
sol_true = sol = odesolver(tt,ic,a)
x_true = sol[1,:];
y_true = sol[2,:];

#Create stochastic data using additive Gaussian noise applied at each data point for the true solution

dist=Normal(0,sigma);

x_model_subset = [x_true[1]];
y_model_subset = [y_true[1]];
for (i, t_i) in enumerate(tt)
    if i%grid_ratio == 0
        x_model_subset = vcat(x_model_subset, x_true[i])
        y_model_subset = vcat(y_model_subset, y_true[i])
    end
end

x_data_sample = zeros(length(x_model_subset));
y_data_sample = zeros(length(y_model_subset));

for (i, x_model_i) in enumerate(x_model_subset)
    x_data_sample[i] = x_model_i + rand(dist)
end

for (i, y_model_i) in enumerate(y_model_subset)
    y_data_sample[i] = y_model_i + rand(dist)
end


Loading the synthetic data set used in the paper "Efficient inference for differential equation models without numerical solvers" 

In [ ]:
#The following sets are the specific synthetic data sets used to produce plots in the paper

t_data = readdlm("t_synthetic_data_LV");
t_model = readdlm("t_synthetic_model_LV");

x_data = readdlm("x_synthetic_data_sample_LV");
y_data = readdlm("y_synthetic_data_sample_LV");

x_model = readdlm("x_synthetic_model_LV");
y_model = readdlm("y_synthetic_model_LV");


In [ ]:
#Construct a B-Spline matrix

spline_order = BSplineKit.BSplineOrder(4)  # This corresponds to cubic splines
B = BSplineKit.BSplineBasis(spline_order, LinRange(t_start, t_end, df-1))

B_matrix = collocation_matrix(B, tt, BSplineKit.Derivative(0), SparseMatrixCSC{Float64})
B_matrix[:,1] = ones(N_t)

#Construct a vector of indices, i_obs, showing the indices of the fine grid of true data, tt, that correspond to values of the synthetic data subset, t

i_obs_x = []
for (i, t_i) in enumerate(t_model)
    if t_i in t_data
        append!(i_obs_x,i)
    end
end

i_obs_y = []
for (i, t_i) in enumerate(t_model)
    if t_i in t_data
        append!(i_obs_y,i)
    end
end


#Construct an A_obs matrix. This matrix constructs a matrix in which each row consists of zeros and a single one. The location of the single one is specificed by the index of the i_obs vector.

t_model = collect(t_model)

function create_A_obs(i_obs)
    A_obs = zeros(Int, length(i_obs), length(t_model))
    for i in range(1, length(i_obs))
        A_obs[i, i_obs[i]] = 1
    end
    A_obs
end

#Construct A_obs, B_obs, and beta_est

 xy_data = vcat(x_data, y_data);
 xy_model = vcat(x_model, y_model);

 A_obs_x = create_A_obs(i_obs_x);
 A_obs_y = create_A_obs(i_obs_y);
 A_obs = A_obs_x;
 B_obs_x = A_obs_x*B_matrix;
 B_obs_y = A_obs_y*B_matrix;
 B_obs = B_obs_x;
 beta_x = pinv(B_obs)*x_data;
 beta_y = pinv(B_obs)*y_data;

#Construct first and second derivative matrix approximations

N = length(x_model)
dt = t_model[end]/(N-1)
D1 = diagm(-1 => -1*ones(N-1), 0 => 0*ones(N-1), 1 => ones(N-1))
D1 = (1 / (2*dt))*D1[1:N, :];
D1[1, 1] = -1/dt;
D1[1, 2] = 1/dt;
D1[N, N-1] = -1/dt;
D1[N, N] = 1/dt;

D1B = D1*B_matrix;

#Loglikelihood functions

function loglhood_sum(w,x_data,y_data,model_x,model_y,a,sigma)
    B_obs_aug_x = vcat(B_obs_x, w*dt*D1B);
    B_obs_aug_y = vcat(B_obs_y, w*dt*D1B);
    
    RHS_term_x = a[1]*model_x - model_x.*model_y;
    RHS_term_y = a[2]*model_x.*model_y - model_y;
   
    x_data_aug = vcat(x_data, w*dt*RHS_term_x);
    y_data_aug = vcat(y_data, w*dt*RHS_term_y);
    
    betas_x = B_obs_aug_x \ x_data_aug;
    betas_y = B_obs_aug_y \ y_data_aug;
    
    g=vcat(A_obs_x*B_matrix*betas_x, A_obs_y*B_matrix*betas_y);
    y1 = vcat(x_data, y_data);
    dist=Normal(0,sigma);  

    Dx = D1B*betas_x;
    Dy = D1B*betas_y;

    y2= vcat(Dx, Dy);
    
    f = vcat(a[1]*model_x - model_x.*model_y, a[2]*model_x.*model_y - model_y);   
    
    e3=loglikelihood(dist, (y1.-g));
    e4 = sum(e3) - w*norm((y2 - f))^2; 
    return e4

end;

#New values of w based on standard deviations of results from previous step

function standard_deviation_data_norms(x_data,y_data,betas_x,betas_y,a,sigma)
    g=vcat(A_obs_x*B_matrix*betas_x, A_obs_y*B_matrix*betas_y);
    y = vcat(x_data, y_data);
    dist=Normal(0,sigma);  

    Dx = D1B*betas_x;
    Dy = D1B*betas_y;

    y2= vcat(Dx, Dy);
    
    f = vcat(a[1]*model_x - model_x.*model_y, a[2]*model_x.*model_y - model_y);
    
    e=std((y.-g));
    return e
end;

function standard_deviation_model_norms(betas_x,betas_y,a,sigma)
    Dx = D1B*betas_x;
    Dy = D1B*betas_y;
                    
    y2= vcat(Dx, Dy);

    model_x = B_matrix*betas_x;
    model_y = B_matrix*betas_y;
    
    f = vcat(a[1]*model_x - model_x.*model_y, a[2]*model_x.*model_y - model_y);
                          
    e=std(w*(y2 - f));
    return e
end;

#Optimising the likelihood

function Optimise(fun,θ₀,lb,ub)    
    tomax=(θ,∂θ)->fun(θ)
    opt=Opt(:LN_NELDERMEAD,length(θ₀))
    opt.max_objective=tomax
    opt.lower_bounds=lb      
    opt.upper_bounds=ub
    opt.maxtime=1*60
    res = optimize(opt,θ₀)
    return res[[2,1]];
end

function funmle_sum(a)
return loglhood_sum(w,x_data,y_data,model_x,model_y,a,sigma);
end



Intial data match - Iteration 0

In [ ]:
#Sum analysis

alphamin= 0.8;
alphamax = 1.3;
betamin=0.8;
betamax=1.3;

#Initial regularisation parameter

w = 0;

alpha_test = 1.1;
beta_test = 0.9;

a_test = [alpha_test, beta_test];

x_model_test = ones(N);
y_model_test = ones(N);

model_x = x_model_test;
model_y = y_model_test;

θG=a_test;
lb=[alphamin,betamin]
ub=[alphamax,alphamax]

#Sum analysis

(xopt_sum,fopt_sum)=Optimise(funmle_sum,θG,lb,ub)
alphamle_sum=xopt_sum[1]
betamle_sum=xopt_sum[2]

#Solving the linear algebra optimisation problem

B_obs_aug_x = vcat(B_obs_x, w*dt*D1B);
B_obs_aug_y = vcat(B_obs_y, w*dt*D1B);

RHS_term_x = a[1]*model_x - model_x.*model_y;
RHS_term_y = a[2]*model_x.*model_y - model_y;

x_data_aug = vcat(x_data, w*dt*RHS_term_x);
y_data_aug = vcat(y_data, w*dt*RHS_term_y);

betas_x_sum = B_obs_aug_x \ x_data_aug;
betas_y_sum = B_obs_aug_y \ y_data_aug;

Dx = D1B*betas_x_sum;
Dy = D1B*betas_y_sum;

y2= vcat(Dx, Dy);

f = vcat(a[1]*model_x - model_x.*model_y, a[2]*model_x.*model_y - model_y);  

B_matrix_betas_x_sum = B_matrix*betas_x_sum;
B_matrix_betas_y_sum = B_matrix*betas_y_sum;



In [ ]:
#Plot Initial Data Match

p1a0=scatter(t_data,x_data,mc=:blue,msc=:blue,label=false,msw=0,ms=4, dpi = 1000);
p1a0=scatter!(t_data,y_data,mc=:green,msc=:green,label=false,msw=0,ms=4);
p1a0=plot!(t_model, B_matrix_betas_x_sum,lw=2,xlabel=L"t",color=:blue,label=false);
p1a0=plot!(t_model, B_matrix_betas_y_sum,lw=2,xlabel=L"t",color=:green,label=false);
p1a0=plot!(xticks=([0, 5, 10, 15],[L"0",L"5", L"10", L"15"]));
p1a0=plot!(yticks=([0, 1, 2, 3],[L"0",L"1", L"2", L"3"]));
p1a0=plot!(xlabel=L"t", ylabel = L"a(t), b(t)",color=:red, xlims=(t_data[1]-1,t_data[end]+1), ylims=(0,3));
p1a0=plot!(xguidefontsize=10, yguidefontsize=10,xtickfontsize=10, ytickfontsize=10, label = false, size = (400,300))
title!("Initial Data Match", titlefontsize=12)

println("w^(0) = "  * string(w))
plot(p1a0)


Iteration 1

In [ ]:
#New weight and parameters

w = 10^(-2);

(xopt_sum,fopt_sum)=Optimise(funmle_sum,θG,lb,ub)
alphamle_sum=xopt_sum[1]
betamle_sum=xopt_sum[2]

a_test = [alphamle_sum, betamle_sum];
betas_x = betas_x_sum;
betas_y = betas_y_sum;

#Solving the linear algebra optimisation problem

B_obs_aug_x_1 = vcat(B_obs_x, w*dt*D1B);
B_obs_aug_y_1 = vcat(B_obs_y, w*dt*D1B);

RHS_term_x_1 = alphamle_sum*model_x - model_x.*model_y;
RHS_term_y_1 = betamle_sum*model_x.*model_y - model_y;

x_data_aug = vcat(x_data, w*dt*RHS_term_x);
y_data_aug = vcat(y_data, w*dt*RHS_term_y);

betas_x_sum_1 = B_obs_aug_x_1 \ x_data_aug;
betas_y_sum_1 = B_obs_aug_y_1 \ y_data_aug;

Dx_1 = D1B*betas_x_sum_1;
Dy_1 = D1B*betas_y_sum_1;

y2= vcat(Dx_1, Dy_1);

f = vcat(alphamle_sum*model_x - model_x.*model_y, betamle_sum*model_x.*model_y - model_y);  

B_matrix_betas_x_sum_1 = B_matrix*betas_x_sum_1;
B_matrix_betas_y_sum_1 = B_matrix*betas_y_sum_1;


In [ ]:
#Plot Iteration 1
p1a1=scatter(t_data,x_data,mc=:blue,msc=:blue,label=false,msw=0,ms=4, dpi = 1000);
p1a1=scatter!(t_data,y_data,mc=:green,msc=:green,label=false,msw=0,ms=4);
p1a1=plot!(t_model, B_matrix_betas_x_sum_1,lw=2,xlabel=L"t",color=:blue,label=false);
p1a1=plot!(t_model, B_matrix_betas_y_sum_1,lw=2,xlabel=L"t",color=:green,label=false);
p1a1=plot!(xticks=([0, 5, 10, 15],[L"0",L"5", L"10", L"15"]));
p1a1=plot!(yticks=([0, 1, 2, 3],[L"0",L"1", L"2", L"3"]));
p1a1=plot!(xlabel=L"t", ylabel = L"a(t), b(t)",color=:red, xlims=(t_data[1]-1,t_data[end]+1), ylims=(0,3));
p1a1=plot!(xguidefontsize=10, yguidefontsize=10,xtickfontsize=10, ytickfontsize=10, label = false, size = (400,300))
title!("Iteration 1", titlefontsize=12)

println("w^(1) = "  * string(w))
plot(p1a1)

Iteration 2

In [ ]:
#New weight and parameters

(xopt_sum,fopt_sum)=Optimise(funmle_sum,θG,lb,ub)
alphamle_sum=xopt_sum[1]
betamle_sum=xopt_sum[2]

a_test = [alphamle_sum, betamle_sum];
betas_x = betas_x_sum_1;
betas_y = betas_y_sum_1;

w = (standard_deviation_data_norms(x_data, y_data, betas_x, betas_y, a_test, sigma))/(standard_deviation_model_norms(betas_x, betas_y, a_test, sigma));

#Solving the linear algebra optimisation problem

B_obs_aug_x_2 = vcat(B_obs_x, w*dt*D1B);
B_obs_aug_y_2 = vcat(B_obs_y, w*dt*D1B);

RHS_term_x_2 = alphamle_sum*model_x - model_x.*model_y;
RHS_term_y_2 = betamle_sum*model_x.*model_y - model_y;

x_data_aug = vcat(x_data, w*dt*RHS_term_x);
y_data_aug = vcat(y_data, w*dt*RHS_term_y);

betas_x_sum_2 = B_obs_aug_x_2 \ x_data_aug;
betas_y_sum_2 = B_obs_aug_y_2 \ y_data_aug;

Dx_2 = D1B*betas_x_sum_2;
Dy_2 = D1B*betas_y_sum_2;

y2= vcat(Dx_2, Dy_2);

f = vcat(alphamle_sum*model_x - model_x.*model_y, betamle_sum*model_x.*model_y - model_y);  

B_matrix_betas_x_sum_2 = B_matrix*betas_x_sum_2;
B_matrix_betas_y_sum_2 = B_matrix*betas_y_sum_2;


In [ ]:
#Plot Iteration 2
p1a2=scatter(t_data,x_data,mc=:blue,msc=:blue,label=false,msw=0,ms=4, dpi = 1000);
p1a2=scatter!(t_data,y_data,mc=:green,msc=:green,label=false,msw=0,ms=4);
p1a2=plot!(t_model, B_matrix_betas_x_sum_2,lw=2,xlabel=L"t",color=:blue,label=false);
p1a2=plot!(t_model, B_matrix_betas_y_sum_2,lw=2,xlabel=L"t",color=:green,label=false);
p1a2=plot!(xticks=([0, 5, 10, 15],[L"0",L"5", L"10", L"15"]));
p1a2=plot!(yticks=([0, 1, 2, 3],[L"0",L"1", L"2", L"3"]));
p1a2=plot!(xlabel=L"t", ylabel = L"a(t), b(t)",color=:red, xlims=(t_data[1]-1,t_data[end]+1), ylims=(0,3));
p1a2=plot!(xguidefontsize=10, yguidefontsize=10,xtickfontsize=10, ytickfontsize=10, label = false, size = (400,300))
title!("Iteration 2", titlefontsize=12)

println("w^(2) = "  * string(w))
plot(p1a2)

Iteration 3

In [ ]:
#New weight and parameters

(xopt_sum,fopt_sum)=Optimise(funmle_sum,θG,lb,ub)
alphamle_sum=xopt_sum[1]
betamle_sum=xopt_sum[2]

a_test = [alphamle_sum, betamle_sum];
betas_x = betas_x_sum_2;
betas_y = betas_y_sum_2;

w = (standard_deviation_data_norms(x_data, y_data, betas_x, betas_y, a_test, sigma))/(standard_deviation_model_norms(betas_x, betas_y, a_test, sigma));

#Solving the linear algebra optimisation problem

B_obs_aug_x_3 = vcat(B_obs_x, w*dt*D1B);
B_obs_aug_y_3 = vcat(B_obs_y, w*dt*D1B);

RHS_term_x_3 = alphamle_sum*model_x - model_x.*model_y;
RHS_term_y_3 = betamle_sum*model_x.*model_y - model_y;

x_data_aug = vcat(x_data, w*dt*RHS_term_x);
y_data_aug = vcat(y_data, w*dt*RHS_term_y);

betas_x_sum_3 = B_obs_aug_x_3 \ x_data_aug;
betas_y_sum_3 = B_obs_aug_y_3 \ y_data_aug;

Dx_3 = D1B*betas_x_sum_3;
Dy_3 = D1B*betas_y_sum_3;

y2= vcat(Dx_3, Dy_3);

f = vcat(alphamle_sum*model_x - model_x.*model_y, betamle_sum*model_x.*model_y - model_y);  

B_matrix_betas_x_sum_3 = B_matrix*betas_x_sum_3;
B_matrix_betas_y_sum_3 = B_matrix*betas_y_sum_3;



In [ ]:
#Plot Iteration 3
p1a3=scatter(t_data,x_data,mc=:blue,msc=:blue,label=false,msw=0,ms=4, dpi = 1000);
p1a3=scatter!(t_data,y_data,mc=:green,msc=:green,label=false,msw=0,ms=4);
p1a3=plot!(t_model, B_matrix_betas_x_sum_3,lw=2,xlabel=L"t",color=:blue,label=false);
p1a3=plot!(t_model, B_matrix_betas_y_sum_3,lw=2,xlabel=L"t",color=:green,label=false);
p1a3=plot!(xticks=([0, 5, 10, 15],[L"0",L"5", L"10", L"15"]));
p1a3=plot!(yticks=([0, 1, 2, 3],[L"0",L"1", L"2", L"3"]));
p1a3=plot!(xlabel=L"t", ylabel = L"a(t), b(t)",color=:red, xlims=(t_data[1]-1,t_data[end]+1), ylims=(0,3));
p1a3=plot!(xguidefontsize=10, yguidefontsize=10,xtickfontsize=10, ytickfontsize=10, label = false, size = (400,300))
title!("Iteration 3", titlefontsize=12)

println("w^(3) = "  * string(w))
plot(p1a3)


In [ ]:
#Create contour plot

M=50;
alpharange=LinRange(alphamin,alphamax,M);
betarange=LinRange(betamin,betamax,M);

df=2
llstar95=-quantile(Chisq(df),0.95)/2
 
colorbar_ticks=([-10, -20, -30],[L"-10", L"-20", L"-30"]);
p2=contourf(alpharange,betarange,(alpharange,betarange)->funmle_sum([alpharange,betarange]) - funmle_sum([alphamle_sum, betamle_sum]),lw=0,xlabel=L"\alpha",ylabel=L"\delta",c=:grays, clims = (-10, 0), colorbar_ticks=colorbar_ticks, colorbar_title = L"\bar{\ell}", colorbar_tickfontsize = 10,dpi = 1000);
p2=contour!(alpharange,betarange,(alpharange,betarange)->funmle_sum([alpharange,betarange]) - funmle_sum([alphamle_sum, betamle_sum]),levels=[llstar95],lw=4,c=:gold,legend=false);
p2=scatter!([alphamle_sum],[betamle_sum],markersize=3,markershape=:circle,markercolor=:red,msw=0,ms=4,label=false);
p2=plot!(xlims=(alphamin,alphamax),xticks=([1, 1.2],[L"1.0", L"1.2"]),ylims=(alphamin,betamax),yticks=([1, 1.2],[L"1.0", L"1.2"]));
p2=plot!(xguidefontsize=12, yguidefontsize=12,xtickfontsize=10, ytickfontsize=10, left_margin = 5mm, bottom_margin = 0mm, top_margin = 5mm, xlabelfontsize=10, ylabelfontsize = 10, size = (400,300))


In [ ]:
#Plot figure and MLE values for alpha and beta

println(L"\hat{\alpha} = "  * string(alphamle_sum))
println(L"\hat{\beta} = "  * string(betamle_sum))
spline_plot = plot(p1a0,p1a3,p2,layout = grid(1, 3), size = (650, 350), dpi = 1000) 
